# HARXHAR Deep Learning RunnerUnified notebook for PatchTSMixer and AE+Ridge GPU backtests.Designed for **cell-by-cell execution** via the `googlecolab/colab-mcp`connector — do **not** use *Runtime > Run All*.| Cell | Tag | Purpose | When to run ||------|-----|---------|-------------|| 1 | setup | Mount Drive, clone repo, install deps | Once per runtime || 2 | parameters | Experiment config — **edit before each run** | Before each run || 3 | validate | Check config + GPU, write status `"validated"` | After editing params || 4 | run | Launch DL experiment (background) | Once per run || 5 | collect | Copy results to Drive | After run succeeds || 6 | eval | QLIKE / MSE / MAE on results | After collect || 7 | status_check | Read status file + GPU utilization | Anytime (polling) |

In [ ]:
# --- Cell 1: Setup ---import os# Mount Google Drive (required for status tracking and result persistence)from google.colab import drivedrive.mount("/content/drive")REPO_URL = "https://github.com/jamesdchen/harxhar.git"REPO_DIR = "/content/harxhar"if not os.path.isdir(REPO_DIR):    !git clone {REPO_URL} {REPO_DIR}else:    !cd {REPO_DIR} && git pull --ff-onlyos.chdir(REPO_DIR)!pip install -q torch transformers numpy pandas scikit-learn tqdm pyarrowfrom src.notebook_utils import verify_gpu, clear_statusverify_gpu()clear_status()print("Setup complete. Drive mounted. Status cleared.")

In [ ]:
# --- Cell 2: Parameters (edit before each run) ---EXPERIMENT = "patchts"       # "patchts" or "ae_ridge"HORIZON = 1                  # forecast horizon (1-48)TRAIN_WINDOW = None          # None = use default from configGPU_COUNT = 1                # Colab typically has 1BATCH_SIZE = None            # None = use defaultEPOCHS = None                # None = use defaultLEARNING_RATE = None         # None = use defaultDATA_PATH = "all30min"RESULTS_DIR = "results_dl"TIMEOUT_HOURS = 2.0          # max runtime before auto-failCHECKPOINT_DIR = None        # set to enable crash recoveryLOSS_LOG_PATH = None         # set to save per-epoch training losses

In [ ]:
# --- Cell 3: Validate config + GPU ---import copyfrom src.notebook_utils import configure_cuda, write_status, get_gpu_utilizationfrom src.core.config import DL_CONFIG, AE_RIDGE_GPU_CONFIGconfigure_cuda()gpu_info = get_gpu_utilization()if EXPERIMENT == "patchts":    config = copy.deepcopy(DL_CONFIG)elif EXPERIMENT == "ae_ridge":    config = copy.deepcopy(AE_RIDGE_GPU_CONFIG)else:    raise ValueError(f"Unknown experiment: {EXPERIMENT!r}. Use 'patchts' or 'ae_ridge'.")config["data_path"] = DATA_PATHconfig["output_path"] = f"{RESULTS_DIR}/{EXPERIMENT}_h{HORIZON}_results.csv"config["gpu_count"] = GPU_COUNTif TRAIN_WINDOW is not None:    config["train_window"] = TRAIN_WINDOWif BATCH_SIZE is not None:    config["train"]["batch_size"] = BATCH_SIZEif EPOCHS is not None:    config["train"]["num_epochs"] = EPOCHSif LEARNING_RATE is not None:    config["train"]["learning_rate"] = LEARNING_RATEif CHECKPOINT_DIR is not None:    config["checkpoint_dir"] = CHECKPOINT_DIRif LOSS_LOG_PATH is not None:    config["loss_log_path"] = LOSS_LOG_PATHwrite_status(    "validated",    experiment=EXPERIMENT,    horizon=HORIZON,    gpu_name=gpu_info.get("gpu_name", "unknown"),    config=config,)print(f"Config OK — {EXPERIMENT}, horizon={HORIZON}, GPU: {gpu_info.get('gpu_name', 'N/A')}")

In [ ]:
# --- Cell 4: Launch experiment (background process) ---import subprocess, os, signalPIDFILE = '/content/harxhar_train.pid'LOGFILE = '/content/harxhar_train.log'# Build CLI command with all parameters from Cell 2/3extra_args = []if BATCH_SIZE is not None:    extra_args += ['--batch-size', str(BATCH_SIZE)]if EPOCHS is not None:    extra_args += ['--epochs', str(EPOCHS)]if LEARNING_RATE is not None:    extra_args += ['--learning-rate', str(LEARNING_RATE)]if TRAIN_WINDOW is not None:    extra_args += ['--train-window', str(TRAIN_WINDOW)]if CHECKPOINT_DIR is not None:    extra_args += ['--checkpoint-dir', CHECKPOINT_DIR]if LOSS_LOG_PATH is not None:    extra_args += ['--loss-log-path', LOSS_LOG_PATH]output_csv = f"{RESULTS_DIR}/{EXPERIMENT}_h{HORIZON}_results.csv"os.makedirs(RESULTS_DIR, exist_ok=True)cmd = [    'python', '-m', 'src.cli.gpu_executor',    '--experiment', EXPERIMENT,    '--input-path', DATA_PATH,    '--output', output_csv,    '--gpu-count', str(GPU_COUNT),    '--horizon', str(HORIZON),    '--timeout-hours', str(TIMEOUT_HOURS),    '--write-status',] + extra_args# Launch in background — kernel stays free for status pollingwith open(LOGFILE, 'w') as log_fh:    proc = subprocess.Popen(        cmd,        stdout=log_fh, stderr=subprocess.STDOUT,        cwd='/content/harxhar',        preexec_fn=os.setsid,    )with open(PIDFILE, 'w') as f:    f.write(str(proc.pid))print(f'Training launched as PID {proc.pid}')print(f'Output: {output_csv}')print(f'Log:    {LOGFILE}')print(f'Kernel is free — use Cell 7 to check progress.')

In [ ]:
# --- Cell 5: Collect results to Drive ---import osimport shutilimport pandas as pdfrom src.notebook_utils import write_statuscsv_path = f"{RESULTS_DIR}/{EXPERIMENT}_h{HORIZON}_results.csv"assert os.path.exists(csv_path), f"Results CSV not found: {csv_path}. Is training finished?"results = pd.read_csv(csv_path)print(f"Loaded {len(results):,} rows from {csv_path}")drive_results_dir = f"/content/drive/MyDrive/harxhar_results/{EXPERIMENT}"os.makedirs(drive_results_dir, exist_ok=True)drive_csv = shutil.copy2(csv_path, drive_results_dir)write_status("collected", local_csv=csv_path, drive_csv=drive_csv, n_results=len(results))print(f"Results copied to Drive: {drive_csv}")

In [ ]:
# --- Cell 6: Evaluate metrics ---from src.evaluation.metrics import calculate_global_metricsfrom src.notebook_utils import print_metrics, write_statusmetrics = calculate_global_metrics(results)print_metrics(metrics)write_status("evaluated", metrics={k: round(v, 6) for k, v in metrics.items()})

In [ ]:
# --- Cell 7: Status check (safe to run anytime) ---import json, os, subprocessfrom src.notebook_utils import read_status, get_gpu_utilizationPIDFILE = '/content/harxhar_train.pid'LOGFILE = '/content/harxhar_train.log'# --- Process liveness ---if os.path.exists(PIDFILE):    pid = int(open(PIDFILE).read().strip())    try:        os.kill(pid, 0)  # signal 0 = check existence only        print(f'Process {pid}: RUNNING')    except ProcessLookupError:        print(f'Process {pid}: FINISHED')    except PermissionError:        print(f'Process {pid}: RUNNING (owned by another user)')else:    print('No PID file — training not launched yet.')# --- Status JSON ---status = read_status()if status:    print('\n=== Status ===')    print(json.dumps(status, indent=2, default=str))else:    print('\nNo status file found.')# --- GPU utilization ---gpu = get_gpu_utilization()print(f"\nGPU: {gpu.get('gpu_name', 'N/A')}")print(f"Utilization: {gpu.get('gpu_util_pct', 'N/A')}%")print(f"Memory: {gpu.get('mem_used_mb', 'N/A')}/{gpu.get('mem_total_mb', 'N/A')} MB")print(f"Temperature: {gpu.get('temp_c', 'N/A')}°C")# --- Log tail ---if os.path.exists(LOGFILE):    print('\n=== Last 30 lines of log ===')    !tail -30 {LOGFILE}